# GDrive Transfer with Rclone and Google Colab
This notebook sets up rclone, clones the repository, and runs the Web-UI to transfer files between cloud drives.

In [1]:
# @title Install Dependencies & Setup Rclone
# @description Install rclone, cloudflared, clone the repository and change directory.
import os
import subprocess
import sys

# Clone repository if not already in the correct directory
if not os.path.exists("src/main.py"):
    print("Cloning repository...")
    subprocess.run("git clone https://github.com/HIKANER/GDrive-transfer.git", shell=True)
    os.chdir("GDrive-transfer")
    print(f"Changed directory to: {os.getcwd()}")

print("Installing rclone...")
subprocess.run("curl https://rclone.org/install.sh | bash", shell=True)

print("Installing cloudflared...")
subprocess.run("curl -L --output cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared.deb && rm cloudflared.deb", shell=True)

print("Installing python dependencies...")
subprocess.run("pip install fastapi uvicorn python-multipart", shell=True)

print("Setup completed successfully!")

Cloning repository...


FileNotFoundError: [Errno 2] No such file or directory: 'GDrive-transfer'

In [ ]:
# @title Run Web-UI & Expose with Cloudflared
# @description Start the FastAPI server and expose it via cloudflared tunnel.
import os
import subprocess
import time
import secrets
import threading
import sys

# Ensure we are in the correct directory
if os.path.exists("GDrive-transfer") and not os.path.exists("src/main.py"):
    os.chdir("GDrive-transfer")

# Generate a random token for authentication
AUTH_TOKEN = secrets.token_hex(16)

# Write the token to a file so main.py can read it or pass it via environment variable
os.environ["AUTH_TOKEN"] = AUTH_TOKEN

# Start FastAPI server in the background
print("Starting FastAPI server...")
backend_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "src.main:app", "--host", "127.0.0.1", "--port", "8000"],
    env=os.environ.copy()
)

# Wait for FastAPI to start
time.sleep(3)

# Start cloudflared tunnel in the background
print("Starting cloudflared tunnel...")
cloudflared_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Function to read cloudflared output and extract the public URL
def monitor_tunnel():
    url_found = False
    for line in iter(cloudflared_process.stdout.readline, ''):
        if "trycloudflare.com" in line:
            # Extract URL
            parts = line.split()
            for part in parts:
                if "trycloudflare.com" in part:
                    url = part.strip()
                    if not url.startswith("http"):
                        url = "https://" + url
                    print("\n" + "="*80)
                    print(f" GDrive-transfer Web-UI is live!")
                    print(f" Access URL: {url}/?token={AUTH_TOKEN}")
                    print("="*80 + "\n")
                    url_found = True
                    break
        if url_found:
            # Keep printing output or break
            pass

# Start monitoring thread
threading.Thread(target=monitor_tunnel, daemon=True).start()

try:
    # Keep the cell running
    while True:
        if backend_process.poll() is not None:
            print("FastAPI server stopped.")
            break
        if cloudflared_process.poll() is not None:
            print("Cloudflared tunnel stopped.")
            break
        time.sleep(1)
except KeyboardInterrupt:
    print("Stopping services...")
    backend_process.terminate()
    cloudflared_process.terminate()
    print("Services stopped.")